# Notebook 02 -- Model Training, Evaluation, and Risk Scoring

**Objective**: Train a dropout classifier on the engineered feature table, evaluate performance, extract feature importance, compute per-student risk scores, assign burnout categories, identify behavioural triggers, generate intervention recommendations, and export all artifacts.

**Input**: `/content/artifacts/engineered_features.csv` (produced by Notebook 01)

**Outputs**:
- `/content/artifacts/dropout_model.joblib` (serialized sklearn pipeline)
- `/content/artifacts/metrics.json` (accuracy, ROC-AUC, classification report)
- `/content/artifacts/predictions.csv` (per-student risk score, triggers, interventions)

In [ ]:
# ── Section 1: Configuration ────────────────────────────────────────────────────
from pathlib import Path
import warnings
import json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED  = 42
ARTIFACTS_DIR = Path("/content/artifacts")
FEATURE_PATH  = ARTIFACTS_DIR / "engineered_features.csv"

assert FEATURE_PATH.exists(), (
    "engineered_features.csv not found. Run Notebook 01 first."
)

df = pd.read_csv(FEATURE_PATH)
print("Loaded:", FEATURE_PATH)
print("Shape :", df.shape)
print("Target distribution:", dict(df["Dropout"].value_counts()))

## Section 2: Feature/Target Split and Preprocessing Setup

Exclude non-predictive columns (`Student_ID`, `synthetic_feedback`) and the target (`Dropout`). Numeric features receive median imputation; categorical features receive most-frequent imputation followed by one-hot encoding.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import joblib

# Define target and features
TARGET = "Dropout"
EXCLUDED = {"Student_ID", "synthetic_feedback", TARGET}
feature_cols = [c for c in df.columns if c not in EXCLUDED]

X = df[feature_cols]
y = df[TARGET].astype(int)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Features: {len(feature_cols)} ({len(num_cols)} numeric, {len(cat_cols)} categorical)")
print(f"Target  : {TARGET} | 1-rate = {y.mean():.2%}")

## Section 3: Train/Test Split and Model Training

Stratified 80/20 split to preserve class balance. The model is a `RandomForestClassifier` (400 trees, balanced class weights) wrapped inside a full sklearn `Pipeline` with the column transformer.

In [ ]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot",  OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ]
)

# Full model pipeline
clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        random_state=RANDOM_SEED,
        class_weight="balanced",
        n_jobs=-1,
    )),
])

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED,
)

# Train
clf.fit(X_train, y_train)
print(f"Training complete | Train size: {len(X_train)} | Test size: {len(X_test)}")

## Section 4: Model Evaluation

Compute accuracy, ROC-AUC, and full classification report on the held-out test set.

In [ ]:
# Predictions
y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

# Metrics
acc     = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
report  = classification_report(y_test, y_pred, output_dict=True)

print("=" * 50)
print("MODEL EVALUATION RESULTS")
print("=" * 50)
print(f"  Accuracy : {acc:.4f}")
print(f"  ROC-AUC  : {roc_auc:.4f}")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Dropout", "Dropout"]))

## Section 5: Confusion Matrix and ROC Curve

Visual diagnostics: confusion matrix to inspect false positive/negative rates, and ROC curve to visualize threshold tradeoff.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["No Dropout", "Dropout"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False,
)
axes[0].set_title("Confusion Matrix")

# ROC curve
RocCurveDisplay.from_predictions(
    y_test, y_proba, ax=axes[1], name="RandomForest",
)
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[1].set_title(f"ROC Curve (AUC = {roc_auc:.3f})")

plt.tight_layout()
plt.show()

## Section 6: Feature Importance (Key Behavioural Triggers)

Extract and rank features by their contribution to the model's predictions. These serve as the "key behavioural triggers" required by the problem statement.

In [ ]:
# ── 6 · Feature-importance bar chart ──────────────────────────
ohe_feature_names = (
    pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)
importances = pipeline.named_steps["clf"].feature_importances_

feat_imp = (
    pd.DataFrame({"feature": ohe_feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
)

TOP_N = 15
top = feat_imp.head(TOP_N).sort_values("importance")

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top["feature"], top["importance"], color="#2563eb")
ax.set_xlabel("Gini Importance")
ax.set_title(f"Top {TOP_N} Features (Behavioural Triggers)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.3f}"))
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/feature_importance.png", dpi=150)
plt.show()

# Store ordered feature list for later trigger extraction
top_global_features = feat_imp["feature"].tolist()
print(f"\n✅ Top {TOP_N} features saved  ·  chart exported to {ARTIFACT_DIR}/feature_importance.png")

## Section 7: Per-Student Risk Scoring

Apply the trained model to **every** student in the feature table to produce:
| Column | Description |
|--------|-------------|
| `dropout_probability` | Raw model probability (0–1) |
| `risk_score` | Scaled 0–100 |
| `burnout_risk_level` | Low / Medium / High |

In [ ]:
# ── 7 · Score every student ────────────────────────────────────
def risk_level(score: float) -> str:
    if score <= 33:
        return "Low"
    elif score <= 66:
        return "Medium"
    return "High"

# Predict on the FULL feature table (not just test set)
X_full = df[FEATURE_COLS]
probas_full = pipeline.predict_proba(X_full)[:, 1]

predictions = df[["Student_ID"]].copy()
predictions["dropout_probability"] = probas_full.round(4)
predictions["risk_score"]          = (probas_full * 100).round(1)
predictions["burnout_risk_level"]  = predictions["risk_score"].apply(risk_level)

# Distribution summary
print("🔢 Risk-level distribution:")
print(predictions["burnout_risk_level"].value_counts().to_string())
print(f"\n📊 Mean risk score: {predictions['risk_score'].mean():.1f}")
predictions.head(10)

## Section 8: Key Behavioural Triggers per Student

For each student, identify the **top-5 features** that contributed most to their individual prediction using the global feature importances as a proxy. A feature "fires" for a student when its transformed value deviates notably from the cohort median.

In [ ]:
# ── 8 · Per-student behavioural triggers ──────────────────────
preprocessor = pipeline.named_steps["preprocessor"]
clf          = pipeline.named_steps["clf"]

X_transformed = preprocessor.transform(X_full)
# Convert sparse matrix to dense if needed
if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

feature_names = preprocessor.get_feature_names_out()
importances   = clf.feature_importances_

# Compute per-student weighted deviation from cohort median
medians = np.median(X_transformed, axis=0)
deviations = np.abs(X_transformed - medians)  # (n_students, n_features)
weighted   = deviations * importances          # weight by global importance

TOP_K = 5

def extract_triggers(row_idx: int) -> str:
    """Return comma-separated top-K trigger names for one student."""
    top_idxs = np.argsort(weighted[row_idx])[::-1][:TOP_K]
    # Clean OHE prefixes for readability
    names = [feature_names[i].split("__")[-1] for i in top_idxs]
    return ", ".join(names)

predictions["key_behavioural_triggers"] = [
    extract_triggers(i) for i in range(len(predictions))
]

print(f"✅ Top-{TOP_K} triggers assigned to each student")
predictions[["Student_ID", "burnout_risk_level", "key_behavioural_triggers"]].head(10)

## Section 9: Recommended Intervention Strategy

Map each student's triggers and risk level to an actionable intervention using a rule-based engine:

| Trigger keyword | Intervention |
|-----------------|-------------|
| `Assignment_Completion` / `Delay` | Academic advisor + deadline extension |
| `Attendance` | Engagement counselling |
| `Stress` / `Sentiment` | Well-being referral |
| `Study_Hours` / `GPA` | Peer tutoring programme |
| *Fallback* | General academic support check-in |

In [ ]:
# ── 9 · Rule-based intervention recommendation ───────────────
INTERVENTION_RULES: list[tuple[list[str], str]] = [
    (["assignment", "completion", "delay"],
     "Schedule academic-advisor meeting; consider deadline extension"),
    (["attendance", "absent"],
     "Engagement counselling + attendance monitoring plan"),
    (["stress", "sentiment", "feedback"],
     "Refer to student well-being / mental-health services"),
    (["study_hours", "gpa", "grade"],
     "Enrol in peer-tutoring or supplemental-instruction programme"),
    (["social", "extracurricular"],
     "Connect with student-life coordinator for social engagement"),
]

def recommend_intervention(triggers: str, risk_level: str) -> str:
    """Return the first matching intervention, or a generic fallback."""
    triggers_lower = triggers.lower()
    for keywords, action in INTERVENTION_RULES:
        if any(kw in triggers_lower for kw in keywords):
            if risk_level == "High":
                return f"🔴 URGENT — {action}"
            return action
    # Fallback
    if risk_level == "High":
        return "🔴 URGENT — General academic-support check-in with counsellor"
    return "General academic-support check-in"

predictions["recommended_intervention_strategy"] = predictions.apply(
    lambda r: recommend_intervention(r["key_behavioural_triggers"], r["burnout_risk_level"]),
    axis=1,
)

print("✅ Interventions assigned\n")
print("📋 Intervention distribution:")
print(predictions["recommended_intervention_strategy"].value_counts().to_string())
predictions[["Student_ID", "burnout_risk_level", "key_behavioural_triggers",
             "recommended_intervention_strategy"]].head(10)

## Section 10: Export Artifacts

Save all deliverables to `/content/artifacts/`:

| File | Contents |
|------|----------|
| `dropout_model.joblib` | Trained sklearn Pipeline (preprocessor + classifier) |
| `metrics.json` | Accuracy, Precision, Recall, F1, ROC-AUC |
| `predictions.csv` | Per-student predictions with risk scores, triggers & interventions |
| `feature_importance.png` | Top-15 feature importance chart |

In [ ]:
# ── 10 · Save all artifacts ────────────────────────────────────
import json, joblib

os.makedirs(ARTIFACT_DIR, exist_ok=True)

# 10-a  Model pipeline
model_path = f"{ARTIFACT_DIR}/dropout_model.joblib"
joblib.dump(pipeline, model_path)
print(f"💾 Model saved → {model_path}  ({os.path.getsize(model_path)/1024:.0f} KB)")

# 10-b  Metrics JSON
metrics_path = f"{ARTIFACT_DIR}/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"💾 Metrics saved → {metrics_path}")

# 10-c  Predictions CSV
csv_cols = [
    "Student_ID",
    "dropout_probability",
    "risk_score",
    "burnout_risk_level",
    "key_behavioural_triggers",
    "recommended_intervention_strategy",
]
pred_path = f"{ARTIFACT_DIR}/predictions.csv"
predictions[csv_cols].to_csv(pred_path, index=False)
print(f"💾 Predictions saved → {pred_path}  ({len(predictions)} rows)")

# ── Final Summary ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  🎓  BehAnalytics – Model Training Complete")
print("=" * 60)
print(f"  Accuracy  : {metrics['accuracy']:.4f}")
print(f"  Precision : {metrics['precision']:.4f}")
print(f"  Recall    : {metrics['recall']:.4f}")
print(f"  F1 Score  : {metrics['f1']:.4f}")
print(f"  ROC-AUC   : {metrics['roc_auc']:.4f}")
print(f"  Students  : {len(predictions)}")
high_risk = (predictions["burnout_risk_level"] == "High").sum()
print(f"  High-Risk : {high_risk}  ({high_risk/len(predictions)*100:.1f}%)")
print("=" * 60)
print(f"\n📁 All artifacts saved to: {ARTIFACT_DIR}/")
print("   └── dropout_model.joblib")
print("   └── metrics.json")
print("   └── predictions.csv")
print("   └── feature_importance.png")